# 02 - LLaMEA Evolutionary Synthesis Pipeline

Multi-process parallel algorithm discovery across the BBOB problem benchmark matrix with SQLite persistence and auto-resumption.

The current configuration selects **f1, f8, f11, f15 and f21**, dimensions **2, 10 and 50**, and noise levels **0, 0.05, 0.1 and 0.2**, with all four prompt strategies in Explicit and Implicit modes. Each candidate receives **100,000 objective calls** and the existing **30-second timeout**. All candidates in a fresh synthesis session use **instance 1**; resumed sessions retain their recorded instance.


**Before running a new study:** complete the manual storage cutover to fresh database/code/checkpoint/result directories and restart the kernel. Editing configurations does not activate the separately prepared staging area, and the current completion checks can otherwise reuse experiments from the old database. The old study is preserved in the verified archive.


In [1]:
import os
import sys
from pathlib import Path

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(root_dir)
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Keep module identities stable for multiprocessing/pickling.
# After updating project code or configuration, restart the kernel instead of evicting modules.

from evolution.infra.di import EvolutionProvider
from evolution.application.campaign.run import SynthesisCampaignCoordinator
from shared.infra.lifecycle import WorkflowScope

print("[OK] LLaMEA Evolution Pipeline initialized.")


[OK] LLaMEA Evolution Pipeline initialized.


## 1. LLM Provider Connection
Initialize and verify the LLM connection configured in your `.env` file.


In [2]:
# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(EvolutionProvider(), context={}, request_context={})
campaign = services.get(SynthesisCampaignCoordinator)
print("=" * 65)
print(f"Model Target: {campaign.model_name}")
print("=" * 65)

Model Target: gwdg.glm-5.3-flash


## 2. Load Configuration & Experiment Matrix
The setup cell defines explicit target problems, dimensions, prompts, noise levels and execution parameters. The audit checks current completion status in SQLite. Setup does not run migrations or launch synthesis. Databases without the optional `synthesis_seed` column remain supported: seeds are not persisted there, but problem/noise seed assignment is unchanged.

The audit and run calls contain explicit study parameters; optional experiment selections belong to the run call. Pass dynamic requests to `run_campaign(...)` in the execution cell: `recover=True` discovers interrupted runs and unresolved synthesis failures for the active model only where no completed valid champion exists. `rerun_experiment_ids=[...]` replaces those exact experiment records from generation one, retaining their IDs and deleting old iterations, diagnostics, champion state, code and checkpoints. Stop any other campaign using these IDs first. Automatic recovery resumes one pending record or replaces one failed record without creating new experiment rows; any completed valid champion satisfies automatic recovery, regardless of failed candidate iterations, other failed sessions, or leftover queued records. `resume_experiment_ids=[...]` only resumes those running records and cannot be combined with recovery or replacement. Calling `run_campaign(**synthesis_parameters)` without selections uses normal full-matrix scheduling. Each call is independent; omit manual IDs on later launches after successful repairs. IDs belong to the current database; review them after an import. The audit counts conditions with a valid completed champion; running and failed record counts are history, not reasons to rerun a covered condition. The standard protocol uses 10 candidate iterations in one synthesis session, not 10 independent trials. A valid finite synthesis result may still perform poorly in benchmarking; request a manual rerun to improve it. No migrations are run.

In [3]:
matrix_df, summary = campaign.audit_matrix(
    problem_targets=[{"id": problem, "dimensions": [2, 10, 50]} for problem in [1, 8, 11, 15, 21]],
    noise_conditions=[
        {"std": 0.0, "noise_model": "none"},
        {"std": 0.05, "noise_model": "heteroscedastic"},
        {"std": 0.1, "noise_model": "heteroscedastic"},
        {"std": 0.2, "noise_model": "heteroscedastic"},
    ],
    synthesis_modes=[
        {"mode": mode, "strategies": ["baseline", "thinking", "vectorization", "guided"]}
        for mode in ["explicit", "implicit"]
    ],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
display(matrix_df) if "display" in globals() else print(matrix_df.to_string())



🎯 Synthesis Matrix Audit for 'gwdg.glm-5.3-flash':
   • Configured Matrix Size:  480 conditions
   • Completed (Champions):   0/480 (0.0%)
   • Without Champion:       480
                                                         Target Runs  Completed  Running Records  Failed Records                      Status
Problem         Dimension Environment     Strategy                                                                                          
f1 (Sphere)     2D        Explicit (0.0)  Baseline                 1          0                1               0  🔄 Incomplete (1 to resume)
                                          Thinking                 1          0                1               0  🔄 Incomplete (1 to resume)
                                          Vectorization            1          0                1               0  🔄 Incomplete (1 to resume)
                                          Guided                   1          0                1               0  🔄 Incomp

## 3. Parallel Multi-Process Experiment Execution (With Auto-Resumption & DB Sync)
Automatically queries `data/db.sqlite3` to check which experiments are still pending or incomplete:
1. **Interrupted Runs (`status == 'running'`)**: Resumed only where a champion is still missing, or to fulfill an explicit manual request. Extra queued records do not invalidate an existing champion.
2. **Completed Runs (`status == 'completed'`)**: A finite final champion makes the condition complete regardless of failures among its 10 candidate iterations.
3. **Failed Synthesis Sessions**: Retried automatically only when the condition has no completed valid champion and `retry_failed_synthesis = true`. Poor benchmark performance instead motivates explicit `rerun_experiment_ids`.
4. **Pending Runs**: In full-matrix mode, fresh tasks fill remaining protocol replicates. Recovery mode only resumes/retries eligible conditions and explicit manual requests.

All tasks execute concurrently across worker processes using `ProcessPoolRunner` with safe SQLite WAL concurrency.

In [ ]:
# Run the configured matrix for the connected model, reusing completed experiments.
results = campaign.run_campaign(
    problem_targets=[{"id": problem, "dimensions": [2, 10, 50]} for problem in [1, 8, 11, 15, 21]],
    noise_conditions=[
        {"std": 0.0, "noise_model": "none"},
        {"std": 0.05, "noise_model": "heteroscedastic"},
        {"std": 0.1, "noise_model": "heteroscedastic"},
        {"std": 0.2, "noise_model": "heteroscedastic"},
    ],
    synthesis_modes=[
        {"mode": mode, "strategies": ["baseline", "thinking", "vectorization", "guided"]}
        for mode in ["explicit", "implicit"]
    ],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)



🧬 LLAMEA EVOLUTIONARY ALGORITHM SYNTHESIS
   Model: gwdg.glm-5.3-flash | Pending Tasks: 480 | Concurrency: 8 workers

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere) | Strategy: Baseline | Exp ID: #6299
No archive path provided, standard initialisation.
Initializing first population

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere) | Strategy: Baseline | Exp ID: #6303
No archive path provided, standard initialisation.
Initializing first population

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere) | Strategy: Thinking | Exp ID: #6304

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: clean (σ=0.0) | Problem: f1 (Sphere) | Strategy: Vectorization | Exp ID: #6305
No archive path provided, standard initialisation.
Initializing first population
No archive path provided, standard initialisation.
Initializing first population

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D |

2026-10-27 14:03:19 WARNING Gateway gwdg.glm-5.3-flash: InternalServerError (status 500), attempt 1/4; shared cooldown 10.6s; will retry
2026-10-27 14:03:35 WARNING Gateway gwdg.glm-5.3-flash: InternalServerError (status 500), attempt 1/4; shared cooldown 10.4s; will retry
2026-10-27 14:04:05 WARNING Gateway gwdg.glm-5.3-flash: InternalServerError (status 500), attempt 2/4; shared cooldown 20.9s; will retry



[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strategy: Guided | Exp ID: #6310

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strategy: Baseline | Exp ID: #6307

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strategy: Guided | Exp ID: #6314

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strategy: Vectorization | Exp ID: #6313

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strategy: Thinking | Exp ID: #6312

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strategy: Baseline | Exp ID: #6311

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strategy: Vectorization | Exp ID: #6309

[1/1] 🧬 Task: gwdg.glm-5.3-flash | Dim: 2D | Noise: noisy (σ=0.05) | Problem: f1 (Sphere) | Strat

2026-10-27 14:04:39 WARNING Gateway gwdg.glm-5.3-flash: InternalServerError (status 500), attempt 1/4; shared cooldown 10.3s; will retry
